### Großstadtbauern [Bwinf44 2025](https://bwinf.de/bundeswettbewerb/44/) - A5

[Video](https://youtu.be/KefnpW60ZzA)

<img src='aufgabe.png' width='251'>

#### Eingabedaten

Jede Datei enthält ein Szenario mit Gemüsesorten und Gerichten und ist so aufgebaut:

- Zeile 1: Die Anzahl n der verschiedenen Gemüsesorten.
- Folgende n Zeilen: Jede Zeile enthält eine Gemüsesorte.
- Nächste Zeile: Die Anzahl m der gewünschten Gerichte.
- Folgende m Zeilen: Jede Zeile enthält ein Gericht, bestehend aus drei durch ein Leerzeichen getrennte Gemüsesorten.

Hier eine Beispieleingabe:

    5
    Zwiebel
    Tomate
    Kohlrabi
    Kartoffel
    Zucchini
    3
    Zwiebel Tomate Zucchini
    Kohlrabi Tomate Kartoffel
    Kartoffel Zwiebel Tomate

In diesem Szenario stehen die fünf  Gemüsesorten Zwiebel, Tomate, Kohlrabi, Kartoffel und Zucchini zur Verfügung.
Auf der Wunschliste stehen drei Gerichte. Für das erste Gericht werden Zwiebeln, Tomaten und Zucchini benötigt. Das zweite Gericht besteht aus Kohlrabi, Tomaten und Kartoffeln und für das dritte Gericht werden Kartoffeln, Zwiebeln und Tomaten verwendet.

#### Beispieldaten

In [ ]:
# Anschauen der Beispieldaten 1-6
nr = '1'
print(f'Beispiel {nr}:')
f = open(f'beispieldaten/bauern{nr}.txt',encoding='utf-8')
print(f.read().strip())
f.close()

#### Lösungsidee

Wir haben ein 2D-Array plan gegeben und sollen entscheiden, welche Gemüse in jedes Feld kommt. Wir formulieren constraints und nutzen einen Solver in Minizinc um an Lösungen zu kommen.

<img src='bild1.png' width='701'>

Die Gerichte könnten wir als sets modellieren oder als arrays. Wir nutzen hier arrays, die sind in der Regel besser für die Laufzeit.

<img src='bild2.png' width='501'>

#### Progamm für Beispiel 2

In [ ]:
int: m = 4;
enum Gemuese = {Kartoffel, Zwiebel, Moehre, Reis};
array[1..m, Gemuese] of bool: gericht = array2d(1..4, Gemuese,[true, true, true, false, true, true, false, true, true, false, true, true, false, true, true, true]);

% Erste Zeile: keine Verschiebung
constraint
  forall(i in 0..3)(
    plan[1,3*i+1] = plan[1,3*i+2] /\
    plan[1,3*i+2] = plan[1,3*i+3]
  );

% Zweite Zeile: +1
% mod 12 nimmt die Werte von 0..11 an. Mit +1 erhalten
% wir den Bereich 1..12
constraint
  forall(i in 0..3)(
    plan[2,((3*i+1) mod 12)+1] =
    plan[2,((3*i+2) mod 12)+1] /\
    plan[2,((3*i+2) mod 12)+1] =
    plan[2,((3*i+3) mod 12)+1]
  );

% Dritte Zeile: +2
constraint
  forall(i in 0..3)(
    plan[3,((3*i+2) mod 12)+1] =
    plan[3,((3*i+3) mod 12)+1] /\
    plan[3,((3*i+3) mod 12)+1] =
    plan[3,((3*i+4) mod 12)+1]
  );

% Decision Variablen ------------------------------------

% gekocht[g,j] = Gericht g wird in Monat j gekocht (boolean)
array[1..m, 1..12] of var bool: gekocht;

% plan[i, m] = g - In Anlage i wird im Monat m das Gemüse g angebaut
array[1..3, 1..12] of var Gemuese: plan;

constraint
  forall(g in 1..m, j in 1..12) (    % für jedes gericht g  und jedes monat j
    gekocht[g,j] <->                 % g wird genau dann im monat j gekocht 
      forall(v in Gemuese where gericht[g,v]) (   % wenn für jedes gemüse v, das in gericht g enthalten ist
        exists(i in 1..3)(plan[i,j] = v)          % eine anlage i existiert, die in monat j gemüse v anbaut
      )
  );
  
 
% Zielfunktion: maximiere Zahl der Gerichte, die mindestens einmal gekocht werden
% Summe aller gerichte g, für die ein monat j existiert, in dem das gericht g gekocht wird
solve maximize sum(g in 1..m)( bool2int( exists(j in 1..12)(gekocht[g,j]) ) );  

output[
  "Es koennen " ++
  show(sum(g in 1..m)( bool2int( exists(j in 1..12)(gekocht[g,j])))) ++
  " Gerichte gekocht werden.\n\n"
]
++
[
  show(plan[r,c]) ++ " " ++
  if c = 12 then "\n" else "" endif
  | r in 1..3, c in 1..12
];


#### Eingabe für Minizinc

In [14]:
nr = '3'
f = open(f'beispieldaten/bauern{nr}.txt',encoding='utf-8')

n = int(f.readline())       # Anzahl Gemüsesorten
gemuese = []
for _ in range(n):
    gemuese.append(f.readline().strip())

m = int(f.readline())      # Anzahl Gerichte
gerichte = []
for _ in range(m): 
    gerichte.append(f.readline().split())

print(f'{m=}')          
print(f'{gemuese=}\n')
print(f'{gerichte=}')

m=19
gemuese=['Apfel', 'Avocado', 'Birne', 'Blumenkohl', 'Bohnen', 'Brokkoli', 'Champignons', 'Erbse', 'Erdbeeren', 'Feige', 'Fenchel', 'Gurke', 'Himbeeren', 'Kartoffel', 'Kichererbse', 'Knoblauch', 'Kohlrabi', 'Lauch', 'Linse', 'Mandarine', 'Möhre', 'Olive', 'Paprika', 'Pastinake', 'Pilze', 'Porree', 'Quitte', 'Rhabarber', 'Rosenkohl', 'Rote_Bete', 'Schwarzwurzel', 'Sellerie', 'Spinat', 'Stachelbeere', 'Tomate', 'Weintrauben', 'Zucchini', 'Zwiebel']

gerichte=[['Rosenkohl', 'Tomate', 'Zwiebel'], ['Paprika', 'Schwarzwurzel', 'Spinat'], ['Bohnen', 'Porree', 'Stachelbeere'], ['Kartoffel', 'Pastinake', 'Rosenkohl'], ['Avocado', 'Blumenkohl', 'Kichererbse'], ['Himbeeren', 'Lauch', 'Rosenkohl'], ['Birne', 'Blumenkohl', 'Sellerie'], ['Avocado', 'Schwarzwurzel', 'Sellerie'], ['Möhre', 'Paprika', 'Tomate'], ['Apfel', 'Kartoffel', 'Paprika'], ['Apfel', 'Kohlrabi', 'Paprika'], ['Apfel', 'Kartoffel', 'Tomate'], ['Kartoffel', 'Zwiebel', 'Möhre'], ['Kartoffel', 'Zwiebel', 'Paprika'], ['Kartoffel', 

Wir wollen in Gemüseliste nur Gemüse aufnehmen, das in mindestens einem Gericht vorkommt. Wir wollen die Gemüseliste sortieren, das Sortierkriterium soll sein, wie häufig das Gemüse in einem Gericht vorkommt.

In [15]:
gMap = {}
for ger in gerichte:
    for g in ger:
        if g not in gMap:
            gMap[g] = 1
        else:
            gMap[g]+=1
gemuese = sorted(gMap.keys(), key = lambda x: gMap[x], reverse=True)
print(gMap)
print(gemuese)


{'Rosenkohl': 3, 'Tomate': 3, 'Zwiebel': 3, 'Paprika': 7, 'Schwarzwurzel': 2, 'Spinat': 1, 'Bohnen': 1, 'Porree': 1, 'Stachelbeere': 1, 'Kartoffel': 6, 'Pastinake': 2, 'Avocado': 2, 'Blumenkohl': 2, 'Kichererbse': 1, 'Himbeeren': 1, 'Lauch': 1, 'Birne': 1, 'Sellerie': 2, 'Möhre': 2, 'Apfel': 3, 'Kohlrabi': 1, 'Zucchini': 3, 'Linse': 2, 'Erdbeeren': 2, 'Mandarine': 1, 'Weintrauben': 2, 'Rhabarber': 1}
['Paprika', 'Kartoffel', 'Rosenkohl', 'Tomate', 'Zwiebel', 'Apfel', 'Zucchini', 'Schwarzwurzel', 'Pastinake', 'Avocado', 'Blumenkohl', 'Sellerie', 'Möhre', 'Linse', 'Erdbeeren', 'Weintrauben', 'Spinat', 'Bohnen', 'Porree', 'Stachelbeere', 'Kichererbse', 'Himbeeren', 'Lauch', 'Birne', 'Kohlrabi', 'Mandarine', 'Rhabarber']


Wir wollen auch die Gerichte sortieren. Wenn ein Gericht aus Gemüsen besteht, das in vielen Gerichten vorkommt, dann vermuten wir, dass es wahrscheinlicher ist, dass dieses Gericht bei einer optimalen Belegung zubereitet werden kann.

In [25]:
def score(gericht):
    return sum([gMap[gem] for gem in gericht])

gerichte.sort(key=lambda x : score(x), reverse=True)

for ger in gerichte:
    print(ger, score(ger))
 

['Kartoffel', 'Zwiebel', 'Paprika'] 25
['Apfel', 'Kartoffel', 'Paprika'] 19
['Kartoffel', 'Zwiebel', 'Möhre'] 19
['Kartoffel', 'Zucchini', 'Paprika'] 19
['Kartoffel', 'Radieschen', 'Zwiebel'] 18
['Banane', 'Kohlrabi', 'Paprika'] 16
['Banane', 'Kohlrabi', 'Zwiebel'] 16
['Rosenkohl', 'Tomate', 'Zwiebel'] 15
['Möhre', 'Paprika', 'Tomate'] 15
['Apfel', 'Kohlrabi', 'Paprika'] 15
['Erbse', 'Mirabelle', 'Zwiebel'] 14
['Banane', 'Fenchel', 'Zwiebel'] 14
['Linsen', 'Zucchini', 'Paprika'] 14
['Mais', 'Zwiebel', 'Möhre'] 14
['Paprika', 'Schwarzwurzel', 'Spinat'] 13
['Knoblauch', 'Mirabelle', 'Zwiebel'] 13
['Paprika', 'Quitte', 'Weizen'] 13
['Kartoffel', 'Pastinake', 'Rosenkohl'] 13
['Apfel', 'Kartoffel', 'Tomate'] 13
['Granatapfel', 'Reis', 'Mirabelle'] 11
['Granatapfel', 'Reis', 'Rhabarber'] 10
['Kiwi', 'Pastinake', 'Pflaume'] 10
['Himbeeren', 'Reis', 'Spinat'] 10
['Kiwi', 'Olive', 'Spinat'] 10
['Granatapfel', 'Reis', 'Maracuja'] 9
['Kichererbse', 'Kiwi', 'Mais'] 9
['Linse', 'Olive', 'Reis'] 9
[

Für diese Reihenfolgen erstellen wir jetzt das 2d-Array gericht.

In [ ]:
gericht = []
for ger in gerichte:
    for g in gemuese:
        if g in ger:
            gericht.append('true')
        else:
            gericht.append('false')
print(f'gericht = array2d(1..{len(gerichte)}, Gemuese,{repr(gericht).replace("'","")});')

#### Schreiben der Eingabedatei

In [33]:
def score(gericht):
    return sum([gMap[gem] for gem in gericht])

nr = '6'
f = open(f'beispieldaten/bauern{nr}.txt',encoding='utf-8')

n = int(f.readline())       # Anzahl Gemüsesorten
gemuese = []
for _ in range(n):
    gemuese.append(f.readline().strip())

m = int(f.readline())      # Anzahl Gerichte
gerichte = []
for _ in range(m): 
    gerichte.append(f.readline().split())

gMap = {}
for ger in gerichte:
    for g in ger:
        if g not in gMap:
            gMap[g] = 1
        else:
            gMap[g]+=1
gemuese = sorted(gMap.keys(), key = lambda x: gMap[x], reverse=True)
gerichte.sort(key=lambda x : score(x), reverse=True)

gericht = []
for ger in gerichte:
    for g in gemuese:
        if g in ger:
            gericht.append('true')
        else:
            gericht.append('false') 

f = open('bauer.dzn',encoding='utf-8',mode='w')    
print(f'm = {m};',file=f)
print(f'Gemuese = {repr(gemuese).replace("'","").replace("ö","oe")};',file = f)
print(f'gericht = array2d(1..{len(gerichte)}, Gemuese,{repr(gericht).replace("'","")});',file=f)
f.close()



Mit zusätzlichen constraints können wir Symmetrien brechen. 

In [ ]:
constraint plan[1,1] <= plan[1,4];
constraint plan[1,1] <= plan[1,7];
constraint plan[1,1] <= plan[1,10];

#### Beispiel 4

Bei Beispiel 4 wird nach ca 3 Minuten eine Lösung mit 9 Gerichten gefunden. Das Programm würde vermutlich noch sehr lange laufen, bis klar ist, dass das die optimale Lösung ist.

In [ ]:
Es koennen 9 Gerichte gekocht werden.

Zwiebel Zwiebel Zwiebel Paprika Paprika Paprika Rosenkohl Rosenkohl Rosenkohl Zucchini Zucchini Zucchini 
Paprika Banane Banane Banane Apfel Apfel Apfel Pastinake Pastinake Pastinake Paprika Paprika 
Kartoffel Kartoffel Kohlrabi Kohlrabi Kohlrabi Kartoffel Kartoffel Kartoffel Linsen Linsen Linsen Kartoffel 
----------

Wir können Gerichte ausschließen und dadurch das Problem verkleinern, indem wir ein Gericht an den Anfang setzen und ermitteln wieviele Lösungen damit noch möglich sind. Wenn es weniger als 9 sind, kann dieses Gericht nicht in einer optimalen Lösung vorkommen. Wir müssen dazu alle 6 möglichen Reihenfolgen der Gemüsesorten für dieses Gericht betrachten

In [ ]:
gerMap = {tuple(ger):0 for ger in gerichte}
for ger in gerichte:
    for g in ger:
        gerMap[tuple(ger)]=sum([gMap[g] for g in ger])
gerMap      

In [30]:
g = (('Erdbeeren', 'Mandarine', 'Weintraube'))
print(f'constraint plan[1,1] = {gemuese.index(g[0])+1};')
print(f'constraint plan[2,1] = {gemuese.index(g[1])+1};')
print(f'constraint plan[3,1] = {gemuese.index(g[2])+1};')

constraint plan[1,1] = 27;
constraint plan[2,1] = 40;
constraint plan[3,1] = 28;


#### Beispiel 5

In [ ]:
Es koennen 9 Gerichte gekocht werden.

J J J W W W J J J T T T 
N V V V C C C S S S N N 
Z Z L L L B B B U U U Z 
----------
(3 min)

#### Beispiel 6

In [34]:
g = (('P', 'A', 'B'))
print(f'constraint plan[1,1] = {gemuese.index(g[0])+1};')
print(f'constraint plan[2,1] = {gemuese.index(g[1])+1};')
print(f'constraint plan[3,1] = {gemuese.index(g[2])+1};')

constraint plan[1,1] = 2;
constraint plan[2,1] = 1;
constraint plan[3,1] = 3;


Den anderen constraint nehmen wir wieder heraus.

#### Das Minizinc Programm

In [ ]:
int: m;
enum Gemuese;
array[1..m, Gemuese] of bool: gericht; 


% Erste Zeile: keine Verschiebung
constraint
  forall(i in 0..3)(
    plan[1,3*i+1] = plan[1,3*i+2] /\
    plan[1,3*i+2] = plan[1,3*i+3]
  );

% Zweite Zeile: +1
% mod 12 nimmt die Werte von 0..11 an. Mit +1 erhalten
% wir den Bereich 1..12
constraint
  forall(i in 0..3)(
    plan[2,((3*i+1) mod 12)+1] =
    plan[2,((3*i+2) mod 12)+1] /\
    plan[2,((3*i+2) mod 12)+1] =
    plan[2,((3*i+3) mod 12)+1]
  );

% Dritte Zeile: +2
constraint
  forall(i in 0..3)(
    plan[3,((3*i+2) mod 12)+1] =
    plan[3,((3*i+3) mod 12)+1] /\
    plan[3,((3*i+3) mod 12)+1] =
    plan[3,((3*i+4) mod 12)+1]
  );

% Decision Variablen ------------------------------------

% gekocht[g,j] = Gericht g wird in Monat j gekocht (boolean)
array[1..m, 1..12] of var bool: gekocht;

% plan[i, m] = g - In Anlage i wird im Monat m das Gemüse g angebaut
array[1..3, 1..12] of var Gemuese: plan;

constraint
  forall(g in 1..m, j in 1..12) (    % für jedes gericht g  und jedes monat j
    gekocht[g,j] <->                 % g wird genau dann im monat j gekocht 
      forall(v in Gemuese where gericht[g,v]) (   % wenn für jedes gemüse v, das in gericht g enthalten ist
        exists(i in 1..3)(plan[i,j] = v)          % eine anlage i existiert, die in monat j gemüse v anbaut
      )
  );
  
 
% Zielfunktion: maximiere Zahl der Gerichte, die mindestens einmal gekocht werden
% Summe aller gerichte g, für die ein monat j existiert, in dem das gericht g gekocht wird
solve maximize sum(g in 1..m)( bool2int( exists(j in 1..12)(gekocht[g,j]) ) );  

% constraint plan[1,1] = 27;
% constraint plan[2,1] = 28;
% constraint plan[3,1] = 40;

output[
  "Es koennen " ++
  show(sum(g in 1..m)( bool2int( exists(j in 1..12)(gekocht[g,j])))) ++
  " Gerichte gekocht werden.\n\n"
]
++
[
  show(plan[r,c]) ++ " " ++
  if c = 12 then "\n" else "" endif
  | r in 1..3, c in 1..12
];

